# 전체 모델 비교와 패턴별 결과

주 비교 임계값0.5. AP는 같은 점수에서 임계값에 따라 바뀌지 않습니다. SAML 라벨로 임계값을 고르지 않습니다.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, HTML
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "saml_d_experiments"]
ROOT = next((p.resolve() for p in candidates if (p / "PACKAGE.json").exists()
             and json.loads((p / "PACKAGE.json").read_text())["package"] == "saml_d_experiments"), None)
if ROOT is None:
    raise FileNotFoundError("압축을 풀고 해당 패키지의 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(ROOT / "src"))
print("패키지:", ROOT)


In [ ]:
results = pd.read_csv(ROOT/'results/saml_experiment_comparison.csv')
display(results)
display(pd.read_csv(ROOT/'results/fusion_generalization/전체성능.csv'))

In [ ]:
patterns = pd.read_csv(ROOT/'results/common23/evaluation/saml/pattern_metrics.csv')
fraud = patterns[patterns.positive > 0].pivot(index='pattern',columns='model',values='recall')
normal = patterns[patterns.normal > 0].pivot(index='pattern',columns='model',values='FPR')
display(100*fraud)
display(100*normal)
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(14,7))
(100*fraud).plot.barh(ax=axes[0],title='Fraud Recall (%)')
(100*normal).plot.barh(ax=axes[1],title='Normal FPR (%)')
fig.tight_layout();plt.show()

## 외부 전체 점수로 재평가(선택)

`metadata.json`과 거래 ID를 포함한 입력 명세는 재실행 문서에 있습니다. 저장 결과를 새 실행 결과와 섞지 않습니다. 모델별 최종 logit을 준비해야 하며 확률을 logit으로 잘못 간주하면 안 됩니다.

In [ ]:
RUN_EXTERNAL_EVALUATION = False
EXTERNAL = ROOT/'external_data/saml_scores'
INCLUSIVE = False  # Multi 단독: >0; 결합모델: >=0 이므로 True
from saml_evaluation import evaluate_external
if RUN_EXTERNAL_EVALUATION:
    from datetime import datetime, timezone
    output = ROOT/'outputs'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    result, by_pattern = evaluate_external(EXTERNAL, output, inclusive=INCLUSIVE)
    display(pd.DataFrame([result]));display(by_pattern)
else:
    print('기본 모드: 저장된 실제 결과만 조회했습니다.')

공통23은 오탐을 줄였지만 AP/Recall은 낮아졌고 Normal_Group 오탐은 오히려 커졌습니다. 대체값/입력 제외만으로 일반화가 회복됐다고 결론짓지 않습니다. [최종 분석과 아직 수행하지 않은 후속 계획](../docs/03_결과분석_후속계획.md).